# U24 — Scalable AI Pipelines: Lab

### Real-world brief: an MLOps pipeline for predictive maintenance

A factory wants to predict machine failures. Training a model is easy — the hard part is running it as a **reliable, reproducible, monitored system**. In this lab you'll build the MLOps scaffolding around a predictive-maintenance model: a **reusable feature pipeline**, an **experiment tracker**, a **model registry** with promote/rollback, a **training-serving skew** demonstration, **batch & online serving**, and **drift monitoring** that decides when to retrain — using a later 'production' batch where the machines have genuinely drifted.

**Resources provided:** `machine_health_train.csv` (labelled training data) and `machine_health_live.csv` (a later production batch with drift). Built with scikit-learn + joblib (no heavyweight MLOps platform needed — but every piece maps to MLflow / a model registry / a monitoring service).

_Phase F — ML Engineering / MLOps._

#objectives

Build a reusable feature pipeline (fit once, apply at train & serve)

Track experiments and pick the best run reproducibly

Register, version, promote and roll back models

Diagnose training-serving skew

Serve batch & online predictions, and monitor for data drift

#how to use this lab

Worked demos teach the pattern; 🧪 LAB EXERCISE cells are real tasks — replace `# YOUR CODE HERE`. Run top to bottom with Shift+Enter.

In [ ]:
# === SETUP: build the source files if missing ===
import os
import numpy as np
import pandas as pd


def _make_split(rng, n, drift=False):
    machine_type = rng.choice(["L", "M", "H"], n, p=[0.5, 0.3, 0.2])
    air_temp = rng.normal(298, 2, n)
    process_temp = air_temp + rng.normal(10, 1, n)
    rot_speed = rng.normal(1500, 150, n)
    torque = np.clip(rng.normal(40, 10, n), 4, None)
    tool_wear = rng.uniform(0, 250, n)

    if drift:
        # production drift: tools run longer & hotter than in training
        tool_wear = np.clip(tool_wear + rng.normal(60, 20, n), 0, 320)
        process_temp = process_temp + rng.normal(3.0, 0.5, n)
        torque = torque + rng.normal(4, 1, n)

    # failure: a sharp function of genuine stress drivers (learnable)
    drive = (0.015 * np.maximum(tool_wear - 180, 0) + 0.05 * np.maximum(torque - 50, 0)
             + 0.04 * np.maximum(process_temp - 312, 0)
             + 0.002 * np.abs(rot_speed - 1500))
    thr = np.quantile(drive, 0.80) if not drift else 0.42   # ~20% failure in train
    p = 1 / (1 + np.exp(-3.0 * (drive - thr)))
    failure = (rng.random(n) < p).astype(int)

    return pd.DataFrame({
        "machine_type": machine_type,
        "air_temp_k": air_temp.round(2),
        "process_temp_k": process_temp.round(2),
        "rot_speed_rpm": rot_speed.round(0),
        "torque_nm": torque.round(2),
        "tool_wear_min": tool_wear.round(1),
        "failure": failure,
    })


def build_pdm(train_path="machine_health_train.csv", live_path="machine_health_live.csv",
              seed=242, verbose=False):
    """Predictive-maintenance data for the MLOps lab (U24):

      machine_health_train.csv   labelled training data (~6000 rows)
      machine_health_live.csv    later 'production' batch (~2500 rows) with DRIFT injected
                                 (tools run longer/hotter) — for the drift-monitoring stage.
    """
    rng = np.random.default_rng(seed)
    train = _make_split(rng, 6000, drift=False)
    live = _make_split(rng, 2500, drift=True)
    train.to_csv(train_path, index=False)
    live.to_csv(live_path, index=False)
    if verbose:
        print("train:", train.shape, "| failure rate:", round(train.failure.mean(), 3))
        print("live :", live.shape, "| failure rate:", round(live.failure.mean(), 3))
        print("tool_wear mean  train vs live:", round(train.tool_wear_min.mean(), 1),
              "vs", round(live.tool_wear_min.mean(), 1), "(drift)")
    return train, live

if not (os.path.exists('machine_health_train.csv') and os.path.exists('machine_health_live.csv')):
    build_pdm(); print('Generated source files.')
else:
    print('Found the provided source files.')

In [ ]:
import pandas as pd, numpy as np, json, joblib, time
train = pd.read_csv('machine_health_train.csv')
live = pd.read_csv('machine_health_live.csv')
NUM = ['air_temp_k', 'process_temp_k', 'rot_speed_rpm', 'torque_nm', 'tool_wear_min']
CAT = ['machine_type']
TARGET = 'failure'
print('train:', train.shape, '| live:', live.shape)
print('train failure rate:', round(train.failure.mean(), 3))
train.head(3)

#1. A reusable feature pipeline

In [ ]:
# -----------------------------------------------------------
# 🔹 1A. ONE transformer, fit on TRAIN, reused everywhere (no skew)
# -----------------------------------------------------------
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score

def make_pipeline(model):
    pre = ColumnTransformer([('num', StandardScaler(), NUM),
                             ('cat', OneHotEncoder(handle_unknown='ignore'), CAT)])
    return Pipeline([('prep', pre), ('model', model)])

X = train[NUM + CAT]; y = train[TARGET]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
pipe = make_pipeline(RandomForestClassifier(n_estimators=200, random_state=0)).fit(Xtr, ytr)
pred = pipe.predict(Xte); proba = pipe.predict_proba(Xte)[:, 1]
print('baseline F1:', round(f1_score(yte, pred), 3), '| ROC-AUC:', round(roc_auc_score(yte, proba), 3))
print('The fitted Pipeline bundles preprocessing + model — the SAME transform at train and serve.')

#### 🧪 EXERCISE 1 — Add an engineered feature
1. Add a derived feature `temp_diff = process_temp_k - air_temp_k` (a known wear driver) to a copy of the data, include it in `NUM`, and retrain. Did F1 improve?
2. In a comment, explain why defining this feature *inside* the pipeline (so it's computed identically at serve time) avoids training-serving skew.

In [ ]:
# 1. add temp_diff, retrain, compare F1
# YOUR CODE HERE

# 2. why compute features inside the pipeline: ...   (comment)

#2. Experiment tracking

In [ ]:
# -----------------------------------------------------------
# 🔹 2A. A MINI EXPERIMENT TRACKER (params + metrics + artifact)
# -----------------------------------------------------------
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

runs = []
def log_run(name, model, params):
    p = make_pipeline(model).fit(Xtr, ytr)
    pr = p.predict(Xte); pb = p.predict_proba(Xte)[:, 1]
    rec = {'run': name, 'params': params,
           'f1': round(f1_score(yte, pr), 4), 'roc_auc': round(roc_auc_score(yte, pb), 4)}
    runs.append(rec); return p

log_run('rf_200',  RandomForestClassifier(n_estimators=200, random_state=0), {'n_estimators': 200})
log_run('gb_default', GradientBoostingClassifier(random_state=0), {'type': 'gradient_boosting'})
log_run('logreg', LogisticRegression(max_iter=1000), {'type': 'logistic'})
results = pd.DataFrame(runs).sort_values('roc_auc', ascending=False).reset_index(drop=True)
print(results[['run', 'f1', 'roc_auc']].to_string(index=False))
best_run = results.iloc[0]['run']
print('\nbest run by ROC-AUC:', best_run)

#### 🧪 EXERCISE 2 — Sweep & compare
1. Add three more runs sweeping RandomForest `max_depth` (e.g. 3, 6, 12). Log each.
2. Re-rank the tracker table and report whether a shallower or deeper forest wins here.
3. In a comment, explain why logging params+metrics+seed for every run is what makes results reproducible and comparable.

In [ ]:
# 1-2. sweep max_depth, re-rank the runs table
# YOUR CODE HERE

# 3. why track every run: ...   (comment)

#3. Model registry — version, promote, roll back

In [ ]:
# -----------------------------------------------------------
# 🔹 3A. A FILE-BASED MODEL REGISTRY with stages
# -----------------------------------------------------------
os.makedirs('registry', exist_ok=True)
REG_FILE = 'registry/registry.json'
def load_registry():
    return json.load(open(REG_FILE)) if os.path.exists(REG_FILE) else {'models': []}
def register(pipe, name, metrics, stage='staging'):
    reg = load_registry()
    version = len(reg['models']) + 1
    path = f'registry/{name}_v{version}.joblib'
    joblib.dump(pipe, path)
    reg['models'].append({'name': name, 'version': version, 'stage': stage,
                          'path': path, 'metrics': metrics})
    json.dump(reg, open(REG_FILE, 'w'), indent=2)
    return version

# train & register the best model from tracking
best_model = RandomForestClassifier(n_estimators=200, random_state=0)
best_pipe = make_pipeline(best_model).fit(Xtr, ytr)
m = {'f1': round(f1_score(yte, best_pipe.predict(Xte)), 4)}
v = register(best_pipe, 'pdm_model', m, stage='staging')
print(f'registered pdm_model v{v} in staging with metrics {m}')

In [ ]:
# -----------------------------------------------------------
# 🔹 3B. PROMOTE to production (and keep rollback ability)
# -----------------------------------------------------------
def set_stage(name, version, stage):
    reg = load_registry()
    for mdl in reg['models']:
        if mdl['name'] == name and mdl['stage'] == 'production' and stage == 'production':
            mdl['stage'] = 'archived'        # demote the current prod model
        if mdl['name'] == name and mdl['version'] == version:
            mdl['stage'] = stage
    json.dump(reg, open(REG_FILE, 'w'), indent=2)
def production_model(name):
    reg = load_registry()
    cand = [m for m in reg['models'] if m['name'] == name and m['stage'] == 'production']
    return joblib.load(cand[-1]['path']) if cand else None

set_stage('pdm_model', v, 'production')
print('current registry stages:', [(m['version'], m['stage']) for m in load_registry()['models']])
print('production model loads:', production_model('pdm_model') is not None)

#### 🧪 EXERCISE 3 — A bad deploy & a rollback
1. Register a **deliberately weak** v2 (e.g. `LogisticRegression` or a depth-1 tree) and promote it to production.
2. 'Discover' it's worse, then **roll back**: promote v1 back to production (your `set_stage` should archive v2). Confirm `production_model` now loads the good model again.
3. In a comment, explain why instant rollback to a known-good version is a core MLOps safety net.

In [ ]:
# 1-2. register weak v2, promote, then roll back to v1
# YOUR CODE HERE

# 3. why rollback matters: ...   (comment)

#4. Training-serving skew

In [ ]:
# -----------------------------------------------------------
# 🔹 4A. THE #1 SILENT BUG: transforming serve data differently
# -----------------------------------------------------------
from sklearn.preprocessing import StandardScaler
rf = RandomForestClassifier(n_estimators=200, random_state=0)
sc = StandardScaler().fit(Xtr[NUM])              # scaler fitted on TRAIN
rf.fit(sc.transform(Xtr[NUM]), ytr)
# serve on the later (drifted) production batch
Xlive_num = live[NUM]; ylive = live[TARGET]
# RIGHT: reuse the TRAIN scaler -> drift signal (high tool_wear/temp) is preserved
f1_right = f1_score(ylive, rf.predict(sc.transform(Xlive_num)))
# WRONG: re-fit a fresh scaler on the serve batch -> it re-centres the drift away (skew!)
sc_wrong = StandardScaler().fit(Xlive_num)
f1_wrong = f1_score(ylive, rf.predict(sc_wrong.transform(Xlive_num)))
print(f'F1 reusing train scaler (correct): {f1_right:.3f}')
print(f'F1 re-fitting scaler at serve (skew): {f1_wrong:.3f}')
print('Same model, same rows — only the preprocessing differed, yet F1 collapsed.')
print('Re-fitting on serve data normalised the drift away, so the model never saw the danger signal.')

#### 🧪 EXERCISE 4 — Unseen category skew
1. The training-encoder used `handle_unknown='ignore'`. Create a serve row with a `machine_type` value never seen in training (e.g. `'X'`) and confirm the saved pipeline still predicts without crashing.
2. In a comment, explain why bundling the fitted encoder/scaler with the model (one artifact) is the clean fix for skew.

In [ ]:
# 1. predict on an unseen machine_type using the saved pipeline
# YOUR CODE HERE

# 2. why ship transformer + model together: ...   (comment)

#5. Serving — batch & online

In [ ]:
# -----------------------------------------------------------
# 🔹 5A. ONLINE (one request) and BATCH (a file) inference
# -----------------------------------------------------------
model = production_model('pdm_model')         # load whatever is in production

def predict_online(record: dict):
    row = pd.DataFrame([record])[NUM + CAT]
    p = model.predict_proba(row)[0, 1]
    return {'failure_risk': round(float(p), 3), 'alert': bool(p > 0.5)}

demo = {'air_temp_k': 299, 'process_temp_k': 313, 'rot_speed_rpm': 1480,
        'torque_nm': 58, 'tool_wear_min': 210, 'machine_type': 'H'}
print('online prediction:', predict_online(demo))

def predict_batch(df):
    out = df.copy(); out['failure_risk'] = model.predict_proba(df[NUM + CAT])[:, 1]
    return out
scored = predict_batch(live.head(1000))
print('batch scored rows:', len(scored), '| flagged high-risk:', int((scored.failure_risk > 0.5).sum()))

#### 🧪 EXERCISE 5 — Latency & a simple cache
1. Time 1000 calls to `predict_online`. Report average latency per call in milliseconds.
2. Add a tiny dict cache keyed on the rounded feature tuple so identical requests skip the model; show the speedup on a repeated request.
3. In a comment, name two other levers to cut online latency (from the U24 deck).

In [ ]:
# 1-2. latency measurement + a prediction cache
# YOUR CODE HERE

# 3. other latency levers: ...   (comment)

#6. Drift monitoring & retraining

In [ ]:
# -----------------------------------------------------------
# 🔹 6A. DATA DRIFT via Population Stability Index (PSI)
# -----------------------------------------------------------
def psi(expected, actual, bins=10):
    qs = np.quantile(expected, np.linspace(0, 1, bins + 1))
    qs[0], qs[-1] = -np.inf, np.inf
    e = np.histogram(expected, qs)[0] / len(expected) + 1e-6
    a = np.histogram(actual, qs)[0] / len(actual) + 1e-6
    return float(np.sum((a - e) * np.log(a / e)))

print('PSI (train vs live production batch) — >0.2 = significant drift:')
drift = {f: round(psi(train[f], live[f]), 3) for f in NUM}
for f, v in sorted(drift.items(), key=lambda x: -x[1]):
    flag = 'DRIFT' if v > 0.2 else 'ok'
    print(f'  {f:18s} PSI={v:.3f}  [{flag}]')
drifted = [f for f, v in drift.items() if v > 0.2]
print('\nfeatures that drifted:', drifted)

In [ ]:
# -----------------------------------------------------------
# 🔹 6B. The drift hurts performance -> retrain on fresh data
# -----------------------------------------------------------
Xl, yl = live[NUM + CAT], live[TARGET]
f1_stale = f1_score(yl, model.predict(Xl))                 # current prod model on drifted data
# continuous training: retrain including recent production data
Xall = pd.concat([train[NUM + CAT], live[NUM + CAT].iloc[:1500]])
yall = pd.concat([train[TARGET], live[TARGET].iloc[:1500]])
retrained = make_pipeline(RandomForestClassifier(n_estimators=200, random_state=0)).fit(Xall, yall)
f1_retrained = f1_score(yl.iloc[1500:], retrained.predict(Xl.iloc[1500:]))
f1_old_holdout = f1_score(yl.iloc[1500:], model.predict(Xl.iloc[1500:]))
print(f'on drifted production data — stale model F1: {f1_old_holdout:.3f}')
print(f'after retraining on recent data  F1: {f1_retrained:.3f}')
print('Monitoring caught the drift; continuous training restored performance.')

#### 🧪 EXERCISE 6 — Close the loop
1. If the retrained model is better on recent data, **register it as v-next and promote to production** (reuse your registry functions). Confirm `production_model` now returns the new one.
2. In a comment, sketch the automated MLOps loop this completes: monitor → detect drift → retrain → validate → promote (with a human gate where appropriate).

In [ ]:
# 1. register & promote the retrained model
# YOUR CODE HERE

# 2. the closed MLOps loop: ...   (comment)

#📘 Summary

| Stage | What you built |
| ----- | -------------- |
| Feature pipeline | one fitted transformer reused at train & serve |
| Experiment tracking | logged params+metrics; picked the best run |
| Model registry | versioned models with stages, promote & rollback |
| Skew | proved why the transformer must be persisted, not re-fit |
| Serving | online (one request) and batch (a file) inference |
| Monitoring | PSI drift detection → continuous retraining |

**Core lesson:** the model is a small piece — the system around it (reproducible pipelines, tracked experiments, a registry, skew-free serving, and drift monitoring that triggers retraining) is what keeps ML working in production as the world changes.

**Next — U25:** a different learning paradigm entirely — agents that learn from reward.